# Tech Challenge Fase 4 — LSTM para Previsão de Ações

**Ação:** ITUB4.SA (Itaú Unibanco — B3)  
**Objetivo:** Prever o preço de fechamento usando redes neurais LSTM  
**Pipeline:** Coleta → Pré-processamento → Modelo LSTM → Avaliação → Salvamento

## 1. Configuração e Imports

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
print(f'TensorFlow: {tf.__version__}')

from src.data.collector import load_or_download
from src.model.preprocessing import prepare_data, build_sequences, WINDOW_SIZE, scale_input, inverse_scale
from src.model.lstm import build_model, train_model, evaluate_model, save_model, load_model

SYMBOL = 'ITUB4.SA'
START_DATE = '2018-01-01'
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('tab10')

## 2. Coleta e Análise Exploratória dos Dados

In [ ]:
df = load_or_download(SYMBOL, start_date=START_DATE, cache=True)
print(f'Período: {df.index[0].date()} → {df.index[-1].date()}')
print(f'Total de registros: {len(df)}')
df.tail()

In [ ]:
df.describe()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(df.index, df['Close'], color='steelblue', linewidth=1)
axes[0].set_title(f'Preço de Fechamento — {SYMBOL}', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Preço (R$)')

axes[1].bar(df.index, df['Volume'], color='gray', alpha=0.6)
axes[1].set_title('Volume Negociado', fontsize=12)
axes[1].set_ylabel('Volume')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig('../reports/figures/price_volume.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
df['retorno_diario'] = df['Close'].pct_change()
df['media_movel_20'] = df['Close'].rolling(20).mean()
df['media_movel_60'] = df['Close'].rolling(60).mean()

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(df.index, df['Close'], label='Fechamento', alpha=0.8)
axes[0].plot(df.index, df['media_movel_20'], label='MM 20 dias', linewidth=1.5)
axes[0].plot(df.index, df['media_movel_60'], label='MM 60 dias', linewidth=1.5)
axes[0].legend()
axes[0].set_title('Preço com Médias Móveis')
axes[0].set_ylabel('Preço (R$)')

axes[1].hist(df['retorno_diario'].dropna(), bins=80, color='steelblue', alpha=0.7, edgecolor='white')
axes[1].set_title('Distribuição dos Retornos Diários')
axes[1].set_xlabel('Retorno')
axes[1].set_ylabel('Frequência')

plt.tight_layout()
plt.savefig('../reports/figures/moving_averages.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Pré-processamento e Criação das Sequências

In [ ]:
X_train, y_train, X_val, y_val, X_test, y_test, scaler = prepare_data(df, window=WINDOW_SIZE)

print(f'Janela temporal: {WINDOW_SIZE} dias')
print(f'Treino:  X={X_train.shape}  y={y_train.shape}')
print(f'Val:     X={X_val.shape}    y={y_val.shape}')
print(f'Teste:   X={X_test.shape}   y={y_test.shape}')

## 4. Construção e Treinamento do Modelo LSTM

In [ ]:
model = build_model(window_size=WINDOW_SIZE)
model.summary()

In [ ]:
history = train_model(model, X_train, y_train, X_val, y_val, epochs=100, batch_size=32)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='Treino')
axes[0].plot(history.history['val_loss'], label='Validação')
axes[0].set_title('Perda (MSE) por Época')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('MSE')
axes[0].legend()

axes[1].plot(history.history['mae'], label='Treino')
axes[1].plot(history.history['val_mae'], label='Validação')
axes[1].set_title('MAE por Época')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('MAE')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/figures/training_history.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Avaliação do Modelo

In [ ]:
metrics, y_true, y_pred = evaluate_model(model, X_test, y_test, scaler)

print('=' * 40)
print('MÉTRICAS NO CONJUNTO DE TESTE')
print('=' * 40)
print(f"MAE:  R$ {metrics['MAE']:.4f}")
print(f"RMSE: R$ {metrics['RMSE']:.4f}")
print(f"MAPE: {metrics['MAPE']:.2f}%")

In [ ]:
n_test = len(df) - len(X_test)
test_dates = df.index[n_test + WINDOW_SIZE:]

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(test_dates, y_true, label='Real', color='steelblue', linewidth=1.5)
ax.plot(test_dates, y_pred, label='Previsto (LSTM)', color='tomato', linewidth=1.5, linestyle='--')
ax.set_title(f'LSTM — Previsão vs. Real ({SYMBOL})', fontsize=14, fontweight='bold')
ax.set_xlabel('Data')
ax.set_ylabel('Preço de Fechamento (R$)')
ax.legend(fontsize=12)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('../reports/figures/predictions_vs_real.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
residuals = y_true.flatten() - y_pred.flatten()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(residuals, bins=50, color='steelblue', alpha=0.7, edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_title('Distribuição dos Resíduos')
axes[0].set_xlabel('Real − Previsto (R$)')

axes[1].scatter(y_pred, residuals, alpha=0.4, color='steelblue', s=15)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_title('Resíduos vs. Previsto')
axes[1].set_xlabel('Previsto (R$)')
axes[1].set_ylabel('Resíduo (R$)')

plt.tight_layout()
plt.savefig('../reports/figures/residuals.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Salvamento do Modelo

In [ ]:
save_model(model, scaler, name='lstm_itub4')
print('Modelo salvo em models/lstm_itub4.keras')
print('Scaler salvo em models/lstm_itub4_scaler.joblib')

## 7. Teste de Carga e Previsão

In [ ]:
model_loaded, scaler_loaded = load_model('lstm_itub4')

ultimos_60 = df['Close'].values[-WINDOW_SIZE:].tolist()
X_input = scale_input(ultimos_60, scaler_loaded)
pred_scaled = model_loaded.predict(X_input, verbose=0)[0][0]
pred_preco = inverse_scale(pred_scaled, scaler_loaded)

print(f'Último preço real:    R$ {df["Close"].iloc[-1]:.2f}')
print(f'Previsão próximo dia: R$ {pred_preco:.2f}')

In [ ]:
previsoes = []
window = list(ultimos_60)

for i in range(5):
    X_input = scale_input(window[-WINDOW_SIZE:], scaler_loaded)
    pred_s = model_loaded.predict(X_input, verbose=0)[0][0]
    pred_v = inverse_scale(pred_s, scaler_loaded)
    previsoes.append(pred_v)
    window.append(pred_v)

print('Previsões para os próximos 5 dias úteis:')
for i, p in enumerate(previsoes, 1):
    print(f'  Dia {i}: R$ {p:.2f}')